# Conservation Testing
This notebook is created to test the number of time points that are needed to integrate over to reduce the error of the KdV PINN plotting the time to train against the error of the model

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
import gc
import pickle

import sys
from pathlib import Path

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

from models import KDV

In [2]:
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
testing_seeds = [42, 72, 83, 27, 81, 174, 9705, 2006, 172, 1969, 1975, 92625, 10000, 2342345, 986689]

CUDA available: True


# One Soliton Testing

In [ ]:
testing_indices_1 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_1_soliton_e_low = []

print("Starting 1-Soliton Experiment Energy (low weight) Experiment...")

for testing_val in testing_indices_1:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons             = 1,
            n_hidden_layers          = 4, 
            n_neurons_per_layer      = 32, 
            activation               = nn.Tanh,
            seed                     = current_seed, 
            verbose                  = False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 50000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = 0, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = testing_val,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 1.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 5.0,    
            w_bc                     = 2.0,    
            w_pde                    = 15.0,
            w_momentum               = 1.0,
            w_energy                 = 0.1,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats, _ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_1_soliton_e_low.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('1_soliton_e_low_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_1_soliton_e_low, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_1_soliton_e_low[-1]['mae_mean']:.6e} ===\n")
        
print("1 Soliton Experiment LOW ENERGY OVER!")

Starting 1-Soliton Experiment Energy (low weight) Experiment...


/home/jairdan/miniconda3/envs/soliton-pinns/lib/python3.14/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/home/jairdan/soliton-pinns/models/kdv_2/loss.py:200: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  losses['total'].append(float(total_loss))


  -> Integrals: 0    | Seed: 42   | Time:  25.86 s | MAE: 3.953666e-03
  -> Integrals: 0    | Seed: 72   | Time:  22.70 s | MAE: 3.859125e-03
  -> Integrals: 0    | Seed: 83   | Time:  24.78 s | MAE: 5.141680e-03
  -> Integrals: 0    | Seed: 27   | Time:  44.38 s | MAE: 1.195796e-03
  -> Integrals: 0    | Seed: 81   | Time:  26.75 s | MAE: 1.666212e-03
  -> Integrals: 0    | Seed: 174  | Time:  25.78 s | MAE: 2.331426e-03
  -> Integrals: 0    | Seed: 9705 | Time:  25.11 s | MAE: 3.251429e-03
  -> Integrals: 0    | Seed: 2006 | Time:  20.26 s | MAE: 2.668197e-03
  -> Integrals: 0    | Seed: 172  | Time:  23.77 s | MAE: 2.111738e-03
  -> Integrals: 0    | Seed: 1969 | Time:  23.17 s | MAE: 1.683855e-03
  -> Integrals: 0    | Seed: 1975 | Time:  34.75 s | MAE: 4.548267e-03
  -> Integrals: 0    | Seed: 92625 | Time:  19.55 s | MAE: 7.758834e-03
  -> Integrals: 0    | Seed: 10000 | Time:  25.67 s | MAE: 2.689907e-03
  -> Integrals: 0    | Seed: 2342345 | Time:  17.88 s | MAE: 7.389488e-03
 

In [5]:
testing_indices_1 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_1_soliton_e_med = []

print("Starting 1-Soliton Experiment Energy (medium weight) Experiment...")

for testing_val in testing_indices_1:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons             = 1,
            n_hidden_layers          = 4, 
            n_neurons_per_layer      = 32, 
            activation               = nn.Tanh,
            seed                     = current_seed, 
            verbose                  = False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 50000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = 0, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = testing_val,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 1.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 5.0,    
            w_bc                     = 2.0,    
            w_pde                    = 15.0,
            w_momentum               = 1.0,
            w_energy                 = 0.5,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats,_ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_1_soliton_e_med.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('1_soliton_e_med_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_1_soliton_e_med, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_1_soliton_e_med[-1]['mae_mean']:.6e} ===\n")
        
print("1 Soliton Experiment MEDIUM ENERGY OVER!")

Starting 1-Soliton Experiment Energy (medium weight) Experiment...
  -> Integrals: 0    | Seed: 42   | Time:  26.21 s | MAE: 3.604576e-03
  -> Integrals: 0    | Seed: 72   | Time:  22.67 s | MAE: 3.859125e-03
  -> Integrals: 0    | Seed: 83   | Time:  24.68 s | MAE: 5.141680e-03
  -> Integrals: 0    | Seed: 27   | Time:  44.32 s | MAE: 1.195796e-03
  -> Integrals: 0    | Seed: 81   | Time:  26.78 s | MAE: 1.666212e-03
  -> Integrals: 0    | Seed: 174  | Time:  25.75 s | MAE: 2.331426e-03
  -> Integrals: 0    | Seed: 9705 | Time:  25.05 s | MAE: 3.251429e-03
  -> Integrals: 0    | Seed: 2006 | Time:  20.23 s | MAE: 2.668197e-03
  -> Integrals: 0    | Seed: 172  | Time:  23.76 s | MAE: 2.111738e-03
  -> Integrals: 0    | Seed: 1969 | Time:  23.12 s | MAE: 1.683855e-03
  -> Integrals: 0    | Seed: 1975 | Time:  34.71 s | MAE: 4.548267e-03
  -> Integrals: 0    | Seed: 92625 | Time:  19.55 s | MAE: 7.758834e-03
  -> Integrals: 0    | Seed: 10000 | Time:  25.66 s | MAE: 2.689907e-03
  -> Int

In [6]:
testing_indices_1 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_1_soliton_e_high = []

print("Starting 1-Soliton Experiment Energy (high weight) Experiment...")

for testing_val in testing_indices_1:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons             = 1,
            n_hidden_layers          = 4, 
            n_neurons_per_layer      = 32, 
            activation               = nn.Tanh,
            seed                     = current_seed, 
            verbose                  = False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 50000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = 0, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = testing_val,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 1.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 5.0,    
            w_bc                     = 2.0,    
            w_pde                    = 15.0,
            w_momentum               = 1.0,
            w_energy                 = 1.0,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats, _ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_1_soliton_e_high.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('1_soliton_e_high_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_1_soliton_e_high, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_1_soliton_e_high[-1]['mae_mean']:.6e} ===\n")
        
print("1 Soliton Experiment HIGH ENERGY OVER!")

Starting 1-Soliton Experiment Energy (high weight) Experiment...
  -> Integrals: 0    | Seed: 42   | Time:  26.40 s | MAE: 3.604576e-03
  -> Integrals: 0    | Seed: 72   | Time:  22.79 s | MAE: 3.859125e-03
  -> Integrals: 0    | Seed: 83   | Time:  24.75 s | MAE: 5.141680e-03
  -> Integrals: 0    | Seed: 27   | Time:  44.45 s | MAE: 1.195796e-03
  -> Integrals: 0    | Seed: 81   | Time:  26.87 s | MAE: 1.666212e-03
  -> Integrals: 0    | Seed: 174  | Time:  25.85 s | MAE: 2.331426e-03
  -> Integrals: 0    | Seed: 9705 | Time:  25.05 s | MAE: 3.251429e-03
  -> Integrals: 0    | Seed: 2006 | Time:  20.25 s | MAE: 2.668197e-03
  -> Integrals: 0    | Seed: 172  | Time:  23.75 s | MAE: 2.111738e-03
  -> Integrals: 0    | Seed: 1969 | Time:  23.10 s | MAE: 1.683855e-03
  -> Integrals: 0    | Seed: 1975 | Time:  34.70 s | MAE: 4.548267e-03
  -> Integrals: 0    | Seed: 92625 | Time:  19.52 s | MAE: 7.758834e-03
  -> Integrals: 0    | Seed: 10000 | Time:  25.64 s | MAE: 2.689907e-03
  -> Integ

In [7]:
testing_indices_1 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_1_soliton_m_low = []

print("Starting 1-Soliton Experiment momentum (low weight) Experiment...")

for testing_val in testing_indices_1:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons             = 1,
            n_hidden_layers          = 4, 
            n_neurons_per_layer      = 32, 
            activation               = nn.Tanh,
            seed                     = current_seed, 
            verbose                  = False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 50000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = testing_val, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = 0,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 1.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 5.0,    
            w_bc                     = 2.0,    
            w_pde                    = 15.0,
            w_momentum               = 0.1,
            w_energy                 = 0.1,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats, _ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_1_soliton_m_low.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('1_soliton_m_low_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_1_soliton_m_low, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_1_soliton_m_low[-1]['mae_mean']:.6e} ===\n")
        
print("1 Soliton Experiment LOW MOMENTUM OVER!")

Starting 1-Soliton Experiment momentum (low weight) Experiment...
  -> Integrals: 0    | Seed: 42   | Time:  26.43 s | MAE: 3.604576e-03
  -> Integrals: 0    | Seed: 72   | Time:  22.77 s | MAE: 3.859125e-03
  -> Integrals: 0    | Seed: 83   | Time:  24.83 s | MAE: 5.141680e-03
  -> Integrals: 0    | Seed: 27   | Time:  44.48 s | MAE: 1.195796e-03
  -> Integrals: 0    | Seed: 81   | Time:  26.84 s | MAE: 1.666212e-03
  -> Integrals: 0    | Seed: 174  | Time:  25.84 s | MAE: 2.331426e-03
  -> Integrals: 0    | Seed: 9705 | Time:  25.14 s | MAE: 3.251429e-03
  -> Integrals: 0    | Seed: 2006 | Time:  20.28 s | MAE: 2.668197e-03
  -> Integrals: 0    | Seed: 172  | Time:  23.83 s | MAE: 2.111738e-03
  -> Integrals: 0    | Seed: 1969 | Time:  23.43 s | MAE: 1.683855e-03
  -> Integrals: 0    | Seed: 1975 | Time:  34.95 s | MAE: 4.548267e-03
  -> Integrals: 0    | Seed: 92625 | Time:  19.51 s | MAE: 7.758834e-03
  -> Integrals: 0    | Seed: 10000 | Time:  25.64 s | MAE: 2.689907e-03
  -> Inte

In [ ]:
testing_indices_1 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_1_soliton_m_med = []

print("Starting 1-Soliton Experiment momentum (medium weight) Experiment...")

for testing_val in testing_indices_1:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons             = 1,
            n_hidden_layers          = 4, 
            n_neurons_per_layer      = 32, 
            activation               = nn.Tanh,
            seed                     = current_seed, 
            verbose                  = False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 50000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = testing_val, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = 0,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 1.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 5.0,    
            w_bc                     = 2.0,    
            w_pde                    = 15.0,
            w_momentum               = 0.5,
            w_energy                 = 0.1,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats, _ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_1_soliton_m_med.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('1_soliton_m_med_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_1_soliton_m_med, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_1_soliton_m_med[-1]['mae_mean']:.6e} ===\n")
        
print("1 Soliton Experiment MEDIUM MOMENTUM OVER!")

Starting 1-Soliton Experiment momentum (medium weight) Experiment...


/home/jairdan/miniconda3/envs/soliton-pinns/lib/python3.14/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/home/jairdan/soliton-pinns/models/kdv_2/loss.py:200: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  losses['total'].append(float(total_loss))


  -> Integrals: 0    | Seed: 42   | Time: 159.08 s | MAE: 3.953666e-03


In [ ]:
testing_indices_1 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_1_soliton_m_high = []

print("Starting 1-Soliton Experiment momentum (high weight) Experiment...")

for testing_val in testing_indices_1:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons             = 1,
            n_hidden_layers          = 4, 
            n_neurons_per_layer      = 32, 
            activation               = nn.Tanh,
            seed                     = current_seed, 
            verbose                  = False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 50000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = testing_val, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = 0,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 1.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 5.0,    
            w_bc                     = 2.0,    
            w_pde                    = 15.0,
            w_momentum               = 0.5,
            w_energy                 = 0.1,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats, _ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_1_soliton_m_high.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('1_soliton_m_high_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_1_soliton_m_high, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_1_soliton_m_high[-1]['mae_mean']:.6e} ===\n")
        
print("1 Soliton Experiment HIGH MOMENTUM OVER!")

In [ ]:
testing_indices_1 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_1_soliton_b_low = []

print("Starting 1-Soliton Experiment momentum and energy (low weight) Experiment...")

for testing_val in testing_indices_1:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons             = 1,
            n_hidden_layers          = 4, 
            n_neurons_per_layer      = 32, 
            activation               = nn.Tanh,
            seed                     = current_seed, 
            verbose                  = False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 50000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = testing_val, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = testing_val,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 1.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 5.0,    
            w_bc                     = 2.0,    
            w_pde                    = 15.0,
            w_momentum               = 0.05,
            w_energy                 = 0.05,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats, _ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_1_soliton_b_low.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('1_soliton_b_low_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_1_soliton_b_low, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_1_soliton_b_low[-1]['mae_mean']:.6e} ===\n")
        
print("1 Soliton Experiment LOW MOMENTUM AND ENERGY OVER!")

In [ ]:
testing_indices_1 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_1_soliton_b_med = []

print("Starting 1-Soliton Experiment momentum and energy (medium weight) Experiment...")

for testing_val in testing_indices_1:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons             = 1,
            n_hidden_layers          = 4, 
            n_neurons_per_layer      = 32, 
            activation               = nn.Tanh,
            seed                     = current_seed, 
            verbose                  = False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 50000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = testing_val, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = testing_val,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 1.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 5.0,    
            w_bc                     = 2.0,    
            w_pde                    = 15.0,
            w_momentum               = 0.25,
            w_energy                 = 0.25,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats, _ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_1_soliton_b_med.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('1_soliton_b_med_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_1_soliton_b_med, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_1_soliton_b_med[-1]['mae_mean']:.6e} ===\n")
        
print("1 Soliton Experiment MEDIUM MOMENTUM AND ENERGY OVER!")

In [ ]:
testing_indices_1 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_1_soliton_b_high = []

print("Starting 1-Soliton Experiment momentum and energy (high weight) Experiment...")

for testing_val in testing_indices_1:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons             = 1,
            n_hidden_layers          = 4, 
            n_neurons_per_layer      = 32, 
            activation               = nn.Tanh,
            seed                     = current_seed, 
            verbose                  = False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 50000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = testing_val, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = testing_val,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 1.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 5.0,    
            w_bc                     = 2.0,    
            w_pde                    = 15.0,
            w_momentum               = 0.5,
            w_energy                 = 0.5,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats, _ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_1_soliton_b_high.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('1_soliton_b_high_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_1_soliton_b_high, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_1_soliton_b_high[-1]['mae_mean']:.6e} ===\n")
        
print("1 Soliton Experiment HIGH MOMENTUM AND ENERGY OVER!")

# Two Soliton Test

In [ ]:
testing_indices_2 = [0, 1, 2, 5, 10, 25, 50]

experiment_results_2_soliton_b_low = []

print("Starting 2-Soliton Experiment momentum and energy (low weight) Experiment...")

for testing_val in testing_indices_2:
    results = {
        'time': [],
        'mae': [],
        'max_error': []
    }
   
    for current_seed in testing_seeds:

        INIT_PARAMS = dict(
            num_solitons=2,
            n_hidden_layers=7, 
            n_neurons_per_layer=62, 
            activation=nn.Tanh,
            seed=current_seed, 
            verbose=False,
        )
        
        TRAIN_PARAMS = dict(
            adam_epochs              = 1000,
            verbose_step             = 100,
            n_collocation            = 100000, 
            n_initial                = 30000,  
            n_boundary               = 10000,
            n_momentum               = testing_val, # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_momentum)
            n_energy                 = testing_val,   # only the t component, the x domain resolution will be the same as n_initial (nx * nt) = (n_initial * n_energy)
            adam_lr                  = 1e-3,   
            lbfgs_lr                 = 2.0,    
            lbfgs_history_size       = 100, 
            lbfgs_version            = 'test', #test is 'old' and anything else will default to a modified version of 'new' from legacy
            adaptive_sampling        = False,   
            logging                  = False, #new parameter, stops loss logging bottleneck for quick training (no loss history)
            verbose                  = False,
        )

        TRAIN_WEIGHTS = dict( #seperated out from the train params
            w_ic                     = 10.0,    
            w_bc                     = 1.0,    
            w_pde                    = 100.0,
            w_momentum               = 0.005,
            w_energy                 = 0.005,
        )
    
        model = KDV(INIT_PARAMS)
        training_stats, _ = model.fit(TRAIN_PARAMS, TRAIN_WEIGHTS)
        error_stats = model.test(nx = 1000, nt=1000, error_type='absolute-normalized')

        print(f"  -> Integrals: {testing_val:<4} | Seed: {current_seed:<4} | Time: {training_stats.time:6.2f} s | MAE: {error_stats.mae:.6e}")

        results['time'].append(training_stats.time)
        results['mae'].append(error_stats.mae)
        results['max_error'].append(error_stats.max_error)
        
        # Explicitly free up GPU memory
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    experiment_results_2_soliton_b_low.append({
        'n_integrals': testing_val,
        'time_mean': float(np.mean(results['time'])),
        'time_std': float(np.std(results['time'])),
        'mae_mean': float(np.mean(results['mae'])),
        'mae_std': float(np.std(results['mae'])),
        'raw_data': results
    })
    
    # Save a backup to disk
    with open('2_soliton_b_low_backup.pkl', 'wb') as f:
        pickle.dump(experiment_results_2_soliton_b_low, f)
    
    print(f"=== Finished n_energy={testing_val} | Avg MAE: {experiment_results_2_soliton_b_low[-1]['mae_mean']:.6e} ===\n")
        
print("2 Soliton Experiment LOW MOMENTUM AND ENERGY OVER!")

# Open and Visualize One Soliton Results

In [ ]:
from dataclasses import dataclass

@dataclass
class Result():
    n_integrals : list[int]
    avg_mae : list[float]
    std_mae : list[float]
    avg_time : list[float]
    std_time : list[float]
    avg_max_err : list[float] #average between groups, of max within groups (groups = n_energy, where each seed is a different result)
    max_max_err : list[float] #max between groups
    std_max_err : list[float]

#Import pickled data object
def reconstruct (file: str):
    with open(file, 'rb') as f:
        soliton_raw = pickle.load(f) #this is a list of the dicts of the experimental results, see above experiments for keys

    n_integrals = []
    avg_mae = []
    std_mae = []
    avg_time = []
    std_time = []
    avg_max_err = [] #average between groups, of max within groups (groups = n_energy, where each seed is a different result)
    max_max_err = [] #max between groups
    std_max_err = []
    

    #Reconstructing usable experiment data
    print("Loaded Experiment Results:\n") #formatting
    for result in soliton_raw:
        n_integrals.append(result['n_integrals'])
        avg_mae.append(result['mae_mean'])
        std_mae.append(result['mae_std'])
        avg_time.append(result['time_mean'])
        std_time.append(result['time_std'])
        avg_max_err.append(np.mean(result['raw_data']['max_error']))
        max_max_err.append(np.max(result['raw_data']['max_error']))
        std_max_err.append(np.std(result['raw_data']['max_error']))
        #printing out values
        print(f"n_integrals: {result['n_integrals']:<4} | Avg MAE: {result['mae_mean']:.6e} | Avg Time: {result['time_mean']:.2e}s | Avg Max Error: {np.mean(result['raw_data']['max_error']):.6e}")
        print(f"---------------| Std MAE: {result['mae_std']:.6e} | Std Time: {result['time_std']:.2e}s | Std Max Error: {np.std(result['raw_data']['max_error']):.6e}")

    result = Result(n_integrals, avg_mae, std_mae, avg_time, std_time, avg_max_err, max_max_err, std_max_err)
    return result

def integrals_v_compute(result):
    plt.errorbar(result.n_integrals, result.avg_time, yerr=result.std_time, color='firebrick', marker='o')
    plt.xlabel('Number of Energy Points (n_energy)')
    plt.ylabel('Mean Time (s)')
    plt.title('Training Time vs. Energy Points')
    plt.grid(True)
    plt.show()

def integrals_v_error(result):
    #Mean Error
    fig, ax = plt.subplots(2, 1)
    ax1, ax2 = ax
    ax1.errorbar(result.n_integrals, result.avg_mae, yerr=result.std_mae, color='seagreen', fmt='o')
    ax1.set(xscale='log', xlabel='Log [Number of Energy Points (n_energy)]', ylabel='Mean Error')
    ax1.grid(True)
    #ax2.plot(n_integrals, avg_max_err, marker='o', color='lightseagreen')
    ax2.errorbar(result.n_integrals, result.avg_max_err, yerr=result.std_max_err, color='lightseagreen', fmt='o')
    ax2.set(xscale='log', xlabel='Log [Number of Energy Points (n_energy)]', ylabel='Max Error')
    ax2.grid(True)
    fig.suptitle('Error Metrics vs. Energy Points')
    plt.show()

def compute_v_error(result):
    plt.errorbar(result.avg_time, result.avg_mae, xerr=result.std_time, yerr=result.std_mae, color='mediumorchid', marker='o', linestyle='None')
    plt.xlabel('Time (s)')
    plt.ylabel('Mean Time (s)')
    plt.title('Training Time vs. Error')
    plt.grid(True)
    plt.show()

#### Number of Energy Integrals v. Compute Time

#### Number of Energy Integrals v. Error Metrics

#### Compute Time v. Error Metrics

# Open and Visualize Two Soliton Results

#### Number of Energy Integrals v. Compute Time

#### Number of Energy Integrals v. Error Metrics

#### Compute Time v. Error Metrics